<table style="width: 100%;">
    <tr style="background-color: transparent;"><td>
        <img src="https://data-88e.github.io/assets/images/blue_text.png" width="250px" style="margin-left: 0;" />
    </td><td>
        <p style="text-align: right; font-size: 10pt;"><strong>Economic Models</strong>, Fall 2025<br>
            Dr. Eric Van Dusen <br>
        Sreeja Apparaju <br>
        Kidong Kim</p></td></tr>
</table>

# Lecture 11: Finance

## This notebook takes a look at some simple tools for looking at the stock market
 - Previously Yahooo finance had a free API for reading in historical data on stocks
 - However the Yahoo API got discontiued
 - An awesome quant made a python package that recreated this functionality by scraping the information
 
Check out the documentation for [Yfinance package](https://pypi.org/project/yfinance/)

 The package - called yfinance is not on the datahub so first we need to install it

In [ ]:
try:
    import yfinance as yf
except:
    !pip install yfinance
    import yfinance as yf

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from datetime import timedelta, date, datetime
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets
from IPython.display import display
import warnings
from datascience import *
warnings.filterwarnings('ignore')

%matplotlib inline

## S&P 500 and the Nasdaq

The yfinance package allows us to download by stock ticker and make a Pandas Dataframe - here we will pull in by the market-wide tickers for the S&P 500 and the Nasdaq

In [ ]:
data_SPNQ = yf.download(("^GSPC", '^IXIC'), start="1993-01-29", end="2022-04-05")

The following section uses the dataframe to build out a new dataframe with returns - the amount earned each day on the previous days close

In [ ]:
data_SN = data_SPNQ.iloc[:, [2,3]]
data_SP =data_SPNQ.iloc[:, 0]
data_NQ = data_SPNQ.iloc[:, 1]
dSP = np.array(len(data_SP)-1)
for i in range(len(data_SP)-1):
    dat = ((data_SP[i] - data_SP[i+1])/data_SP[i])*100
    dSP = np.append(dSP,dat)
dNQ = np.array(len(data_NQ)-1)
for i in range(len(data_NQ)-1):
    dat = ((data_NQ[i] - data_NQ[i+1])/data_NQ[i])*100
    dNQ = np.append(dNQ,dat)
data_SN['SP Returns'] = dSP
data_SN['NQ Returns'] = dNQ

In [ ]:
data_SN.iloc[:,[0,1]].plot(color = ('blue', 'red'), figsize=(10,8), alpha =0.3);

In [ ]:
data_SN[['SP Returns', 'NQ Returns']].iloc[1:].plot(color = ('blue', 'red'), figsize=(10,8), alpha = 0.3);

In [ ]:
data_SN.iloc[:,[0,1]].plot(color = ('blue', 'red'), figsize=(10,8), alpha =0.3);

In [ ]:
data_SN[['SP Returns', 'NQ Returns']].iloc[1:].plot(color = ('blue', 'red'), figsize=(10,8), alpha = 0.3);

## Let's dive deeper into the Yfinance API and and work with the data

First we will define the stocks that we want to look at more closely, and examine what sort of information we can get for each stock.  

In [ ]:
apple_ticker = yf.Ticker("aapl")
msft_ticker = yf.Ticker("msft")
tsla_ticker = yf.Ticker("tsla")
amzn_ticker = yf.Ticker("amzn")
nvidia_ticker = yf.Ticker("nvda")

There is actually a lot of information that yfinance API can provide for any equity.  In the example above we only downloaded the closing price for each of the indexes. 

In [ ]:
msft_ticker.info

## Out of all this info - let's extract the stock prices

This will put the dates, prices, and volumes into a *Pandas* dataframe with the name of the stock

In [ ]:

apple = apple_ticker.history(period="max")
nvidia = nvidia_ticker.history(period="max")

In [ ]:
apple
nvidia

## Lets look at the market for Options for Twitter 
 - This will show us the possible strike dates for different options
 - From short term - this week - to long term - in two years


In [ ]:
apple_ticker.options
nvidia_ticker.options

## Downloading Calls and Puts 
Let's download all of the Calls and Puts for Apple and NVIDIA into two tables 

In [ ]:
# Returns two tables: puts and calls for Apple and NVIDIA
apple_option_chain = apple_ticker.option_chain(apple_ticker.options[0])
apple_call = apple_option_chain.calls
apple_put = apple_option_chain.puts
nvidia_option_chain = nvidia_ticker.option_chain(nvidia_ticker.options[0])
nvidia_call = nvidia_option_chain.calls
nvidia_put = nvidia_option_chain.puts
relevant_columns = ['lastPrice', 'change', 'percentChange', 'volume', 'strike']
apple_put, apple_call = apple_put[relevant_columns], apple_call[relevant_columns]
nvidia_put, nvidia_call = nvidia_put[relevant_columns], nvidia_call[relevant_columns]
apple_put.describe(), apple_call.describe(), nvidia_put.describe(), nvidia_call.describe()

## Let's put these together into joint tables that are joined by the strike price for Apple and NVIDIA

In [ ]:
apple_option = pd.merge(apple_put, apple_call, how='inner', on = "strike")
nvidia_option = pd.merge(nvidia_put, nvidia_call, how='inner', on = "strike")
#apple_option = pd.merge(apple_put, apple_call, how='outer', on = "strike")
#nvidia_option = pd.merge(nvidia_put, nvidia_call, how='outer', on = "strike")
apple_option[12:32]
nvidia_option[12:32]

## Now lets code up a graph of the puts and calls for Apple and NVIDIA

### Understanding Options Pricing Graphs

These graphs visualize the relationship between **strike prices** and **option premiums** for both calls and puts:

**What the graph shows:**
- **Red line (Calls)**: The price to buy a call option at different strike prices
- **Blue line (Puts)**: The price to buy a put option at different strike prices  
- **Green vertical line**: The current stock price

**Key patterns to observe:**

For **Call Options** (right to buy):
- Calls with strike prices **below** the current price are more expensive (they're "in the money")
- As strike price increases above current price, call premiums decrease
- Far out-of-the-money calls (high strikes) are cheapest but least likely to profit

For **Put Options** (right to sell):
- Puts with strike prices **above** the current price are more expensive (they're "in the money")
- As strike price decreases below current price, put premiums decrease
- Far out-of-the-money puts (low strikes) are cheapest but least likely to profit

**Where the lines intersect** near the current price shows the point where calls and puts have similar premiums, reflecting market equilibrium and expected volatility.

In [ ]:
current_apple_p = apple.iloc[-1]['Close']
current_nvidia_p = nvidia.iloc[-1]['Close']
current_apple_p, current_nvidia_p

plt.figure().set_size_inches(15, 5)

plt.title("Apple calls vs puts",  fontsize=15)

plt.plot(apple_option['strike'], apple_option['lastPrice_x'], color='r', label='call', linewidth=3)
plt.plot(apple_option['strike'], apple_option['lastPrice_y'], color='b', label='put', linewidth=3)
plt.axvline(x = current_apple_p, color = 'g', label = 'Current Price', linewidth=4) #Current Price

plt.axis([apple_option['strike'].iloc[0], apple_option['strike'].iloc[-1], 0, max(max(apple_option['lastPrice_x']), max(apple_option['lastPrice_y']))])
plt.legend()

plt.figure().set_size_inches(15, 5)

plt.title("NVIDIA calls vs puts",  fontsize=15)

plt.plot(nvidia_option['strike'], nvidia_option['lastPrice_x'], color='r', label='call', linewidth=3)
plt.plot(nvidia_option['strike'], nvidia_option['lastPrice_y'], color='b', label='put', linewidth=3)
plt.axvline(x = current_nvidia_p, color = 'g', label = 'Current Price', linewidth=4) #Current Price

plt.axis([nvidia_option['strike'].iloc[0], nvidia_option['strike'].iloc[-1], 0, max(max(nvidia_option['lastPrice_x']), max(nvidia_option['lastPrice_y']))])
plt.legend()

## Creating a Reusable Function for Options Data

The following cell defines a **reusable function** that packages all the manual steps we did above into a single, convenient tool.

**What the `option()` function does:**
1. Takes any stock ticker as input
2. Downloads the option chain for the nearest expiration date
3. Separates puts and calls
4. Filters to keep only the most relevant columns (lastPrice, change, percentChange, volume, strike)
5. Merges puts and calls into one table joined by strike price
6. Returns the combined dataframe

This makes it easy to analyze options for any stock with just one line of code!

In [ ]:
def option(ticker):
    option_chain = ticker.option_chain(ticker.options[0])
    put, call = option_chain.puts, option_chain.calls
    relevant_columns = ['lastPrice', 'change', 'percentChange', 'volume', 'strike']
    put, call = put[relevant_columns], call[relevant_columns]
    option_sheet = pd.merge(put, call, how='inner', on = "strike")
    return option_sheet

#option(apple_ticker)
# the ticker argument can be any yfinance Ticker object
# some examples are 

option(msft_ticker)

## QuantStats Package
The same developer made a more recent package that draws on Yfinance but makes a whole set of summary tables 

Check out the documentation for the [QuantStats Package](https://pypi.org/project/QuantStats/)

In [ ]:
try:
    import quantstats as qs
except:
    !pip install quantstats
    import quantstats as qs

In [ ]:
import quantstats as qs

# extend pandas functionality with metrics, etc.
qs.extend_pandas()

# fetch the daily returns for a stock
stock = qs.utils.download_returns('TSLA')

# show sharpe ratio
qs.stats.sharpe(stock)

# or using extend_pandas() :)
stock.sharpe()

### QuantStats can make a "Snapshot" of stock performance

In [ ]:
qs.plots.snapshot(stock, title='Tesla Performance')

## Relevant materials and sources

https://algotrading101.com/learn/yfinance-guide/ <br>
https://pypi.org/ <br>
https://pypi.org/project/QuantStats/

## More Cool Things QuantStats Can Do!

QuantStats has three main modules with powerful features:

### 1. **qs.stats** - Performance Metrics (60+ metrics!)
- **Sharpe Ratio** - Risk-adjusted return measure
- **Sortino Ratio** - Like Sharpe but only penalizes downside volatility
- **Calmar Ratio** - Return relative to maximum drawdown
- **Max Drawdown** - Largest peak-to-trough decline
- **Win Rate** - Percentage of profitable periods
- **Volatility** - Standard deviation of returns
- **Value at Risk (VaR)** - Potential loss at a given confidence level
- **Conditional VaR (CVaR)** - Expected loss beyond VaR
- **Beta** - Correlation with market movements
- **Alpha** - Excess return over benchmark
- **R-Squared** - How much variance is explained by benchmark
- **CAGR** - Compound Annual Growth Rate
- **Kelly Criterion** - Optimal bet sizing
- **Tail Ratio** - Ratio of 95th to 5th percentile returns
- **Profit Factor** - Gross profits / Gross losses
- **Recovery Factor** - Net profit / Max drawdown
- **Ulcer Index** - Measure of downside risk

### 2. **qs.plots** - Visualizations
- **Snapshot** - Comprehensive performance overview (as shown above)
- **Monthly Returns Heatmap** - Color-coded monthly performance
- **Distribution Plot** - Return distribution with normal curve overlay
- **Drawdown Plot** - Underwater equity curve
- **Rolling Sharpe/Sortino** - Time-varying risk metrics
- **Rolling Beta** - How correlation changes over time
- **Rolling Volatility** - Time-varying risk
- **Yearly Returns** - Bar chart of annual performance
- **Histogram** - Return frequency distribution
- **Log Returns** - Cumulative returns on log scale
- **Daily Returns** - Simple returns over time

### 3. **qs.reports** - Complete Tearsheets
- **HTML Reports** - Generate full HTML tearsheet with all metrics and plots
- **Basic Report** - Quick overview with key metrics
- **Full Report** - Comprehensive analysis with all available metrics
- **Metrics-only** - Just the numbers, no plots
- **Plots-only** - Just the visualizations
- **Compare multiple strategies** - Side-by-side analysis

### Extra Features:
- **Benchmark Comparison** - Compare your returns against S&P500, Nasdaq, etc.
- **Interactive Plots** - Can export to Plotly for interactive charts
- **Risk Metrics** - Downside deviation, tail risk, skewness, kurtosis
- **Trade Analysis** - Win/loss ratios, consecutive wins/losses
- **Greeks** - Delta, Gamma, Vega, Theta for options

Let's try some of these!

In [ ]:
# Example 1: Monthly Returns Heatmap
stock = qs.utils.download_returns('AAPL')
qs.plots.monthly_heatmap(stock)

In [ ]:
# Example 2: Compare Multiple Metrics at Once
print("Key Performance Metrics for Apple:")
print(f"Sharpe Ratio: {stock.sharpe():.2f}")
print(f"Sortino Ratio: {qs.stats.sortino(stock):.2f}")
print(f"Max Drawdown: {qs.stats.max_drawdown(stock):.2%}")
print(f"Win Rate: {qs.stats.win_rate(stock):.2%}")
print(f"CAGR: {qs.stats.cagr(stock):.2%}")
print(f"Volatility (Annual): {qs.stats.volatility(stock):.2%}")

In [ ]:
# Example 3: Drawdown Analysis
qs.plots.drawdown(stock)

In [ ]:
# Example 4: Rolling Sharpe Ratio (shows how risk-adjusted performance changes over time)
qs.plots.rolling_sharpe(stock)

In [ ]:
# Example 5: Compare Against Benchmark (S&P 500)
# Download S&P 500 returns for comparison
spy = qs.utils.download_returns('SPY')

# Calculate beta and alpha vs S&P 500
beta = qs.stats.greeks(stock, spy)['beta']
print(f"Apple's Beta vs S&P 500: {beta:.2f}")
print(f"(A beta > 1 means more volatile than market, < 1 means less volatile)")

In [ ]:
# Example 6: Returns Distribution
qs.plots.distribution(stock)

### The Ultimate Feature: Full HTML Report

You can generate a complete tearsheet with ALL metrics and plots saved as an HTML file!

In [ ]:
# Example 7: Generate a complete HTML tearsheet (uncomment to run)
# This creates a professional-looking report with ALL metrics and visualizations
# qs.reports.html(stock, benchmark='SPY', output='apple_tearsheet.html', title='Apple Stock Analysis')

# Or show a basic report in the notebook:
qs.reports.basic(stock, benchmark=spy, title='Apple vs S&P 500')